# Step 0 — IDENTITY & GCN · burst #21 `bn110920546`
**Status: FINALIZED** — approved by VIKAS.

Who this burst is, what the alert record says, and the two facts that shape every later step.

In [1]:
import os, json, glob, hashlib, re, numpy as np
import astropy.io.fits as fits
from astropy.table import Table
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
BURST = "bn110920546"
sha = lambda p: hashlib.sha256(open(p,"rb").read()).hexdigest()
rel = lambda p: os.path.join(ROOT, p)
appr = json.load(open(rel(f"results/sweep106/{BURST}/APPROVALS.json")))
print("repo:", ROOT, "| burst:", BURST)

repo: /Users/salim/Desktop/Projects/SingleRest/Two_Breaks | burst: bn110920546


In [2]:
STEP = "0"

In [3]:
s = appr[STEP]
print(f'step {STEP}: {s["status"]}  by {s["by"]}  {s["utc"]}')
for f in s.get("feedback", []):
    print(f'\n  PI feedback: {f["text"][:400]}')
    if f.get("routed"): print(f'  routed -> {f["routed"][:300]}')

step 0: APPROVED  by VIKAS  2026-08-31T04:20:02Z


## 1. Identity, pinned by trigger number

The catalogue name is **ambiguous**: two GBM triggers on the same day share "GRB 110920A".
Ours is `bn110920546`. Never accept a published value keyed to the name alone.

In [4]:
tcat = glob.glob(rel(f"data/{BURST}/glg_tcat_*"))
if tcat:
    hh = fits.open(tcat[0])[0].header
    print(f'  TRIGTIME (MET) : {hh["TRIGTIME"]}')
    print(f'  RA, Dec        : {hh.get("RA_OBJ")}, {hh.get("DEC_OBJ")}   err {hh.get("ERR_RAD")} deg')
else:
    h1 = fits.open(rel(f"data/{BURST}/glg_tte_n0_{BURST}_v00.fit.gz"))[0].header
    print(f'  TRIGTIME (MET) : {h1.get("TRIGTIME")}  (from TTE; no tcat locally)')
s = Table.read(rel("results/grb_sample.ecsv"), format="ascii.ecsv")
r = s[[str(x).strip()==BURST for x in s["TRIGGER_NAME"]]][0]
print(f'  sample RA/Dec  : {float(r["RA"])}, {float(r["DEC"])}')
print(f'  catalog T90    : {float(r["T90"]):.3f} +/- {float(r["T90_ERROR"]):.3f} s')
print(f'  HAS_LAT        : {bool(r["HAS_LAT"])}')

  TRIGTIME (MET) : 338216745.81222  (from TTE; no tcat locally)
  sample RA/Dec  : 209.82, -27.56
  catalog T90    : 160.771 +/- 5.221 s
  HAS_LAT        : False


## 2. Zero GCN circulars — a result, verified with a positive control

In [5]:
raw = open(rel(f"results/gcn/{BURST}/{BURST}_gcn_raw.txt")).read()
print("\n".join(raw.splitlines()[:16]))

GCN CIRCULAR SWEEP — bn110920546  (Step 0, GCNIntelligence.md)
Generated 2026-08-12 by the 106-burst doc-layer worker. NEW FILE (no prior results/gcn/bn110920546/).

RESULT: n_circulars = 0.  ZERO GCN circulars exist for this burst.
This is a RESULT, not an error (GCNIntelligence.md "Honest absence").

Fetch log (every attempt, with its HTTP status):
  1. https://gcn.gsfc.nasa.gov/other/110920A.gcn3   -> HTTP 404 (216 bytes, error page)
  2. https://gcn.gsfc.nasa.gov/other/110920.gcn3    -> HTTP 404 (215 bytes, error page)
  3. gcn.nasa.gov archive search API
     (https://gcn.nasa.gov/circulars?query=<q>&_data=routes/circulars._archive._index)
       query="110920A"      -> totalItems 0,  queryFallback false
       query="110920"       -> totalItems 0,  queryFallback false
       query="110920546"    -> totalItems 0,  queryFallback false
       query="GRB110920A"   -> totalItems 0,  queryFallback false
     CONTROL (proves the endpoint and its 2011 coverage are live):


## 3. The two facts that bind later steps

**(a) Detector consensus.** The mission catalogue and every published analysis use exactly
n0, n1, n3, b0 — the same set approved here. A later disagreement with the literature therefore
cannot be blamed on detector choice; that door is closed in advance.

**(b) The SAA.** Fermi left the South Atlantic Anomaly ~120 s before the trigger, so every
pre-burst background sits in the activation-decay tail. Verified below from the spacecraft
file, not merely quoted.

In [6]:
ph = glob.glob(rel(f"data/{BURST}/glg_poshist_*"))[0]
d = fits.open(ph)[1].data
h1 = fits.open(rel(f"data/{BURST}/glg_tte_n0_{BURST}_v00.fit.gz"))[0].header
T0 = h1["TRIGTIME"]; t = d["SCLK_UTC"]-T0 if "SCLK_UTC" in d.names else d["SCLK"]-T0
fl = d["FLAGS"] if "FLAGS" in d.names else None
if fl is not None:
    saa = (np.asarray(fl)[:,1] if np.ndim(fl)==2 else (np.asarray(fl)>>1)&1)
    m = (t>-400)&(t<100)
    tr = np.where(np.diff(saa[m].astype(int))!=0)[0]
    print("  SAA flag transitions near the trigger (s rel T0):", np.round(t[m][tr],2)[:5])
print("  => every pre-burst background window lies inside the activation-decay tail.")
print("  Biltzinger+2020 use THIS burst as their showcase that polynomial backgrounds")
print("  can give ambiguous answers here.")

  SAA flag transitions near the trigger (s rel T0): [-121.07]
  => every pre-burst background window lies inside the activation-decay tail.
  Biltzinger+2020 use THIS burst as their showcase that polynomial backgrounds
  can give ambiguous answers here.
